# CENTRALASIA – plot already-computed results

This notebook **only reads** cached results and model checkpoints — no training, no data processing.
Run it while the main notebook is still computing other regions.

## 0. Imports & config

In [ ]:
import os
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"]   = "0"

import sys
sys.path.append(os.path.join(os.getcwd(), '../../'))

import warnings
import torch
import pandas as pd
from pathlib import Path

import massbalancemachine as mbm
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset   import *
from regions.TF_Europe.scripts.plotting  import *
from regions.TF_Europe.scripts.models    import *
from regions.TF_Europe.scripts.utils     import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg    = mbm.EuropeTFConfig()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mbm.utils.seed_all(cfg.seed)
mbm.plots.use_mbm_style()

print(f'Device: {device}')

In [ ]:
# ── paths (mirror the main notebook) ─────────────────────────────────────────
BASE_DIR       = Path(cfg.dataPath) / path_cache / 'TF_REGION'
CACHE_DIR      = BASE_DIR / 'LSTM_cache_TL/Transformer_exp'
CACHE_DIR_RANK_CA = CACHE_DIR / 'ranking_ca_subregions'

MODEL_DATE     = '2026-05-21'

SOURCE_REGIONS = ['CH', 'NOR', 'ISL', 'FR']
TARGET_REGIONS = ['CENTRALASIA']

MONTHLY_COLS   = ['t2m','tp','slhf','sshf','ssrd','fal','str','ELEVATION_DIFFERENCE']
STATIC_COLS    = ['aspect','slope','svf']

FRACS          = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.4]
N_RANDOM_SEEDS = 10

os.makedirs('figures', exist_ok=True)

print('BASE_DIR :', BASE_DIR)
print('Results CSV:', BASE_DIR / 'ranking_ca_subregions_results.csv')

## 1. Load saved results CSV

In [ ]:
results_csv = BASE_DIR / 'ranking_ca_subregions_results.csv'

if not results_csv.exists():
    raise FileNotFoundError(
        f'Results CSV not found at {results_csv}.\n'
        'Make sure the main notebook has saved it before running this cell.'
    )

df_ranking_results_ca_sub = pd.read_csv(results_csv)
print(f'Loaded {len(df_ranking_results_ca_sub)} rows from {results_csv.name}')
df_ranking_results_ca_sub.head()

In [ ]:
# Quick summary: which (ranking_target, test_region) combos are present
df_ranking_results_ca_sub.groupby(
    ['ranking_target', 'test_region', 'model', 'strategy']
).agg(
    n_rows=('pct', 'count'),
    RMSE_a_mean=('RMSE_annual', 'mean'),
    R2_a_mean=('R2_annual', 'mean'),
).round(3)

## 2. `plot_ranking_results_extended` – all subregion / ranking combos

In [ ]:
ranking_combos = [
    ('ranked_by_ca_joint',   'CENTRALASIA'),
    ('ranked_by_ca12_joint', 'CA_12'),
    ('ranked_by_ca3_joint',  'CA_3'),
    ('ranked_by_ca4_joint',  'CA_4'),
]

for model_label in ['transformer']:
    for ranking_label, test_label_match in ranking_combos:
        # Check that we actually have data for this combo before plotting
        mask = (
            (df_ranking_results_ca_sub['ranking_target'] == ranking_label) &
            (df_ranking_results_ca_sub['test_region']    == test_label_match) &
            (df_ranking_results_ca_sub['model']          == model_label)
        )
        if not mask.any():
            print(f'[SKIP] No data for {ranking_label} / {test_label_match} / {model_label}')
            continue

        print(f'Plotting: {ranking_label}  →  {test_label_match}')
        fig = plot_ranking_results_extended(
            df_results      = df_ranking_results_ca_sub,
            ranking_label   = ranking_label,
            test_region     = test_label_match,
            source_regions  = SOURCE_REGIONS,
            n_rand_seeds    = N_RANDOM_SEEDS,
            model_label     = model_label,
            save_path       = BASE_DIR / f'ranking_ca_sub_{ranking_label}_{model_label}',
        )

## 3. `plot_pred_vs_truth_grid` – load model checkpoints, no retraining

This cell requires the model `.pt` files to already exist on disk.
It reconstructs the test dataset from the cached monthly data.

> **Note:** loading monthly data and building the test `Dataset` is fast (no ERA5
> reprocessing needed if cache files are present). The cell will tell you if any
> checkpoint is missing.

In [ ]:
# ── reload only what we need for the pred-vs-truth grid ──────────────────────
# Monthly data (reads from cache; set run_flag=False everywhere)

import geopandas as gpd
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

paths_multi = {
    'EU_US_CANADA': {
        'era5_climate_data':
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     'era5_monthly_averaged_data_EU_US_CANADA.nc'),
        'geopotential_data':
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     'era5_geopotential_pressure_EU_US_CANADA.nc'),
    },
    'HMA': {
        'era5_climate_data':
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     'era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc'),
        'geopotential_data':
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     'era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc'),
    },
}

# Load stake dataframes (needed to build monthly datasets)
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

# O2Region mapping for CENTRALASIA (RGI region 13)
rgi_gdf = gpd.read_file(
    cfg.dataPath / RGI_V6_ROOT / RGI_REGIONS['13']['folder'] / RGI_REGIONS['13']['file']
)[['RGIId', 'O2Region']]

glacier_dict = (
    dfs['13'][['GLACIER','RGIId']].drop_duplicates()
    .merge(rgi_gdf, on='RGIId', how='left')
    .set_index('GLACIER').to_dict('index')
)

print('Stake dataframes loaded.')

In [ ]:
# Build monthly cache (reads from disk if already computed)
run_flag_by_code = {r: False for r in SOURCE_REGIONS + TARGET_REGIONS}

monthly_cache = build_monthly_cache(
    cfg              = cfg,
    dfs              = dfs,
    paths_multi      = paths_multi,
    vois_climate     = ['t2m','tp','slhf','sshf','ssrd','fal','str'],
    vois_topographical = ['aspect','slope','svf'],
    region_codes     = SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code = run_flag_by_code,
)

res_xreg_by_source = {region: monthly_cache[region]
                      for region in SOURCE_REGIONS + TARGET_REGIONS}

months_head_pad = res_xreg_by_source['CH']['months_head_pad']
months_tail_pad = res_xreg_by_source['CH']['months_tail_pad']

print('Monthly cache ready.')

In [ ]:
# Build CENTRALASIA test dataset
df_ca_full = res_xreg_by_source['CENTRALASIA']['data_monthly']
df_ca_aug  = res_xreg_by_source['CENTRALASIA']['data_monthly_aug']

ds_ca_test = build_combined_LSTM_dataset(
    df_loss          = df_ca_full,
    df_full          = df_ca_aug,
    monthly_cols     = MONTHLY_COLS,
    static_cols      = STATIC_COLS,
    months_head_pad  = months_head_pad,
    months_tail_pad  = months_tail_pad,
    normalize_target = True,
    expect_target    = True,
    show_progress    = False,
)
print(f'CENTRALASIA test dataset: {len(ds_ca_test)} sequences')

In [ ]:
# Load best_params from the GS log (same as main notebook)
gs_logs_dir = BASE_DIR / 'logs/Transformer_GS'
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, 'transformer_gs_pooled_2026-05-20.csv'))
best_row = logs_gs_transformer.sort_values('valid_loss').iloc[0]

best_params_gs = {
    'Fm':             int(best_row['Fm']),
    'Fs':             int(best_row['Fs']),
    'd_model':        int(best_row['d_model']),
    'nhead':          int(best_row['nhead']),
    'num_layers':     int(best_row['num_layers']),
    'dim_feedforward':int(best_row['dim_feedforward']),
    'dropout':        float(best_row['dropout']),
    'static_layers':  int(best_row['static_layers']),
    'static_hidden':  (None if pd.isna(best_row['static_hidden'])
                       else int(best_row['static_hidden'])),
    'static_dropout': (None if pd.isna(best_row['static_dropout'])
                       else float(best_row['static_dropout'])),
    'head_dropout':   float(best_row['head_dropout']),
    'lr':             float(best_row['lr']),
    'weight_decay':   float(best_row['weight_decay']),
    'two_heads':      False,
    'loss_spec':      None,
    'T_max':          32,
}
print('best_params_gs loaded:', best_params_gs)

In [ ]:
# Load glacier ranking (for build_assets_from_glacier_list)
df_ranked_ca_joint = pd.read_csv(BASE_DIR / 'glacier_ranking_ca_joint.csv')

# Rebuild gl_subsets (fast – no heavy computation)
gl_subsets_ca_joint = build_glacier_subsets(
    df_ranked    = df_ranked_ca_joint,
    fracs        = FRACS,
    n_random_seeds = N_RANDOM_SEEDS,
    seed         = cfg.seed,
)

print('Glacier subsets rebuilt.')

In [ ]:
# ── pred-vs-truth grid: closest vs random for selected fracs ────────────────
FRACS_TO_PLOT = [10, 15, 20, 30]
mode          = 'joint'
models_dir    = BASE_DIR / 'models/ranking_ca_subregions' / 'CENTRALASIA'

ranking_plot_configs = []
missing_checkpoints  = []

for pct in FRACS_TO_PLOT:
    for strategy_name, glaciers in [
        ('closest',    gl_subsets_ca_joint[pct]['closest']),
        ('random_0',   gl_subsets_ca_joint[pct]['random'][0]['glaciers']),
    ]:
        assets = build_assets_from_glacier_list(
            glaciers          = glaciers,
            df_ranked         = df_ranked_ca_joint,
            res_xreg_by_source = res_xreg_by_source,
            monthly_cols      = MONTHLY_COLS,
            static_cols       = STATIC_COLS,
            cfg               = cfg,
            cache_path        = str(
                CACHE_DIR_RANK_CA /
                f'assets_CENTRALASIA_{mode}_{pct}pct_{strategy_name}.joblib'
            ),
            force_recompute   = False,
            months_head_pad   = months_head_pad,
            months_tail_pad   = months_tail_pad,
            src_regions       = SOURCE_REGIONS,
        )

        model, model_path, info = train_or_load_one_source_model(
            cfg          = cfg,
            key          = f'CENTRALASIA_{pct}pct_{strategy_name}',
            lstm_assets  = assets,
            best_params  = best_params_gs,
            device       = device,
            models_dir   = models_dir,
            prefix       = 'transformer_rank_ca',
            train_flag   = False,   # ← load only, never retrain
            force_retrain= False,
            epochs       = 150,
            date         = MODEL_DATE,
            model_type   = 'transformer',
            verbose      = False,
        )

        if model is None:
            missing_checkpoints.append(f'CENTRALASIA_{pct}pct_{strategy_name}')
            print(f'[MISSING] checkpoint for {pct}% {strategy_name}')
            continue

        label = ('Closest' if strategy_name == 'closest' else 'Random seed 0')
        ranking_plot_configs.append(
            (f'{label} {pct}% ({mode})', model, assets)
        )

if missing_checkpoints:
    print(f'\n⚠ {len(missing_checkpoints)} checkpoint(s) missing – those panels will be skipped.')

print(f'\nBuilt {len(ranking_plot_configs)} plot configs.')

In [ ]:
if ranking_plot_configs:
    plot_pred_vs_truth_grid(
        plot_configs = ranking_plot_configs,
        ds_test      = ds_ca_test,
        device       = device,
        cfg          = cfg,
        ncols        = 4,
        ax_xlim      = (-8, 6),
        ax_ylim      = (-8, 6),
        title        = f'Closest vs random ({mode}) → CENTRALASIA',
        save_path    = f'figures/ranking_ca_{mode}_pred_vs_truth',
        figsize      = (25, 12),
    )
else:
    print('No plot configs available – check model checkpoint paths.')

## 4. (Optional) Quick numeric summary from the CSV

In [ ]:
# Mean RMSE_annual per (test_region, strategy) for the transformer
(
    df_ranking_results_ca_sub
    .query("model == 'transformer'")
    .groupby(['test_region', 'pct', 'strategy'])[['RMSE_annual','RMSE_winter','R2_annual']]
    .mean()
    .round(3)
)